# Exploring Aerosol In-situ Data: PNSD, ACSM, and Filter Absorption Photometer Measurements

## Introducton

This notebook provides a hands-on guide for working with atmospheric aerosol datasets relevant to ACTRIS In-Situ. It focuses on three key types of in-situ data: Particle Number Size Distributions (PNSD), Aerosol Chemical Speciation Monitor (ACSM) data, and Filter Absorption Photometer measurements.

Students will learn how to:
* Search for and extract relevant datasets,
* Join datasets from the same station,
* Perform basic plotting and visualization.

The aim is to equip you with practical skills for accessing, combining, and analyzing aerosol in-situ observational data. A basic understanding of Python, especially libraries such as pandas and matplotlib, is expected to follow the exercises effectively.

### Import libaries

In [1]:
# Import necessary libraries
import json  # For working with JSON files
import requests  # For making HTTP requests
import os  # For handling file system operations

import xarray as xr  # For working with multidimensional datasets
import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical operations

import matplotlib.pyplot as plt  # For plotting and visualization
import matplotlib.dates as mdates  # For formatting dates in plots
import plotly.graph_objects as go  # For creating advanced visualizations
import plotly.express as px  # For simpler visualizations

from tqdm.notebook import tqdm  # For creating progress bars
from datetime import datetime  # For working with date and time

## Particle Number Size Distribution (PNSD)

Particle Number Size Distribution (PNSD) datasets are measured using mobility particle size spectrometers, commonly known as DMPS (Differential Mobility Particle Sizer) or SMPS (Scanning Mobility Particle Sizer).

These measurements describe how aerosol particles are distributed by size in the atmosphere and are key to understanding aerosol sources and processes.

In [2]:
# Fetch metadata for aerosol particle number size distribution at Birkenes II, Norway (facility 9cxe) using the ACTRIS API

i = 0
metadata_archive = []
pbar = tqdm(desc="Fetching metadata pages", unit="page")  # Create a progress bar

while True:
    # Make an HTTP GET request to fetch metadata
    response = requests.get(
        f"https://prod-actris-md2.nilu.no/metadata/facility/9cxe/content/aerosol%20particle%20number%20size%20distribution/page/{i}"
    )
    if not response.json():  # Stop if no more data is available
        break
    metadata_archive += response.json()  # Append metadata to the archive
    i += 1
    pbar.update(1)  # Update the progress bar

pbar.close()

Fetching metadata pages: 0page [00:00, ?page/s]

In [3]:
# Extract content information from metadata
files_list = []
for f in metadata_archive:
    time = f["ex_temporal_extent"]
    inst = {"instrument_model": f["md_actris_specific"]["instrument_model"]}
    combined_dict = {**time, **inst}  # Merge the dictionaries
    files_list.append(combined_dict)

df_content_information = pd.DataFrame.from_records(files_list)
# Display the content information for all datasets
df_content_information

,time_period_begin,time_period_end,instrument_model
0,2010-01-14T00:00:00,2017-01-01T00:00:00,GRIMM/190
1,2009-07-10T00:00:00,2012-01-01T00:00:00,NILU/DMPSmodel2
2,2013-10-03T00:00:00,2013-12-16T00:00:00,NILU/DMPSmodel2
3,2013-12-16T00:00:00,2014-01-01T00:00:00,NILU/DMPSmodel2
4,2012-01-01T00:00:00,2013-08-26T00:00:00,NILU/DMPSmodel2
5,2014-01-01T00:00:00,2023-08-25T00:00:00,NILU/DMPSmodel2


In [4]:
# Extract the dataset URL for one specific dataset
pnsd_ds = xr.open_dataset(metadata_archive[5]['md_distribution_information'][1]['dataset_url'])
pnsd_ds

<xarray.Dataset>
Dimensions:                                                  (time: 83958,
                                                              tbnds: 2,
                                                              metadata_time: 11,
                                                              Location: 1,
                                                              pressure_qc_flags: 2,
                                                              relative_humidity_qc_flags: 2,
                                                              temperature_qc_flags: 2,
                                                              D: 25,
                                                              particle_number_size_distribution_amean_qc_flags: 2,
                                                              particle_number_size_distribution_prec1587_qc_flags: 2,
                                                              particle_number_size_distribution_perc8413_qc_flags: 2)
Coordinates:
  * time                                                     (time) datetime64[ns] ...
  * metadata_time                                            (metadata_time) datetime64[ns] ...
  * Location                                                 (Location) |S64 ...
  * D                                                        (D) float64 10.0...
Dimensions without coordinates: tbnds, pressure_qc_flags,
                                relative_humidity_qc_flags,
                                temperature_qc_flags,
                                particle_number_size_distribution_amean_qc_flags,
                                particle_number_size_distribution_prec1587_qc_flags,
                                particle_number_size_distribution_perc8413_qc_flags
Data variables: (12/20)
    time_bnds                                                (time, tbnds) datetime64[ns] ...
    metadata_time_bnds                                       (metadata_time, tbnds) datetime64[ns] ...
    pressure_qc                                              (Location, pressure_qc_flags, time) float64 ...
    pressure_ebasmetadata                                    (Location, metadata_time) |S64 ...
    relative_humidity_qc                                     (Location, relative_humidity_qc_flags, time) float64 ...
    relative_humidity_ebasmetadata                           (Location, metadata_time) |S64 ...
    ...                                                       ...
    pressure                                                 (Location, time) float64 ...
    relative_humidity                                        (Location, time) float64 ...
    temperature                                              (Location, time) float64 ...
    particle_number_size_distribution_amean                  (D, time) float64 ...
    particle_number_size_distribution_prec1587               (D, time) float64 ...
    particle_number_size_distribution_perc8413               (D, time) float64 ...
Attributes: (12/113)
    Conventions:                                   CF-1.8, ACDD-1.3
    featureType:                                   timeSeries
    title:                                         Particle_number_size_distr...
    keywords:                                      ACTRIS, particle_number_si...
    id:                                            55XV-K3VB.nc
    naming_authority:                              EBAS
    ...                                            ...
    geospatial_lat_units:                          degrees_north
    geospatial_lon_units:                          degrees_east
    comment:                                       {\n    "Data definition": ...
    standard_name_vocabulary:                      CF-1.7, ACDD-1.3
    history:                                       None
    creator_url:                                   ebas.nilu.no

In [ ]:
# Combine datasets with the same instrument model into one xarray Dataset
datasets = []

# Loop through the relevant indices (1 to 5)
for idx in range(1, 6):
    dataset_url = metadata_archive[idx]['md_distribution_information'][1]['dataset_url']
    ds = xr.open_dataset(dataset_url)  # Open the dataset

    # Drop all variables with '_qc' in their names to avoid conflicts
    # This is to clean up datasets before merging
    variables_to_drop = [var for var in ds.data_vars if '_qc' in var]
    ds = ds.drop_vars(variables_to_drop)

    datasets.append(ds)  # Append to the list

# Merge all datasets into one xarray Dataset
combined_ds = xr.concat(datasets, dim='time')

combined_ds

<xarray.Dataset>
Dimensions:                                                  (
                                                              metadata_time: 21,
                                                              Location: 1,
                                                              D: 44,
                                                              time: 122294,
                                                              tbnds: 2)
Coordinates:
  * metadata_time                                            (metadata_time) datetime64[ns] ...
  * Location                                                 (Location) |S64 ...
  * D                                                        (D) float64 10.0...
  * time                                                     (time) datetime64[ns] ...
Dimensions without coordinates: tbnds
Data variables: (12/14)
    time_bnds                                                (time, tbnds) datetime64[ns] ...
    metadata_time_bnds                                       (time, metadata_time, tbnds) datetime64[ns] ...
    pressure_ebasmetadata                                    (time, Location, metadata_time) object ...
    relative_humidity_ebasmetadata                           (time, Location, metadata_time) object ...
    temperature_ebasmetadata                                 (time, Location, metadata_time) object ...
    particle_number_size_distribution_amean_ebasmetadata     (time, D, metadata_time) object ...
    ...                                                       ...
    pressure                                                 (Location, time) float64 ...
    relative_humidity                                        (Location, time) float64 ...
    temperature                                              (Location, time) float64 ...
    particle_number_size_distribution_amean                  (D, time) float64 ...
    particle_number_size_distribution_prec1587               (D, time) float64 ...
    particle_number_size_distribution_perc8413               (D, time) float64 ...
Attributes: (12/114)
    Conventions:                                   CF-1.8, ACDD-1.3
    featureType:                                   timeSeries
    title:                                         Particle_number_size_distr...
    keywords:                                      particle_number_size_distr...
    id:                                            RAWY-M6Z5.nc
    naming_authority:                              EBAS
    ...                                            ...
    geospatial_lat_units:                          degrees_north
    geospatial_lon_units:                          degrees_east
    comment:                                       {\n    "Data definition": ...
    standard_name_vocabulary:                      CF-1.7, ACDD-1.3
    history:                                       None
    creator_url:                                   ebas.nilu.no

In [6]:
# Convert the dataset to a DataFrame for easier manipulation
ds = combined_ds # Choose pnsd_ds or combined_ds
df = ds.particle_number_size_distribution_amean.to_dataframe().reset_index()
df = df.dropna()  # Remove rows with missing values

In [ ]:
z = np.log10(df['particle_number_size_distribution_amean'])
zmin = np.min(z)
zmax = np.max(z)
n = 6
tickvals1 = np.linspace(zmin, zmax, n)
ticktext1 = np.round(10 ** np.linspace(zmin, zmax, n), 1)

fig = go.Figure(
    go.Heatmap(
        x=df["time"],
        y=df["D"],
        z=z,
        zsmooth="best",
        colorscale="Jet",
        showscale=True,
        colorbar=dict(
            title="dN/dlogDp ([1/cm3])",
            tickvals=tickvals1,
            ticktext=ticktext1,
            titleside="right",
            titlefont=dict(size=14, family="Arial, sans-serif"),
        ),
    )
)

fig.update_layout(
    title="DMPS - particle number size distribution - Birkenes II, Norway",
    xaxis_title="Time",
    yaxis_title="D [nm]",
    title_font_size=20,
)
fig.update_yaxes(type="log")
fig.show()

/home/lemu/anaconda3/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/lemu/anaconda3/lib/python3.9/site-packages/numpy/core/function_base.py:151: RuntimeWarning: invalid value encountered in multiply
  y *= step
/home/lemu/anaconda3/lib/python3.9/site-packages/numpy/core/function_base.py:161: RuntimeWarning: invalid value encountered in add
  y += start
